# Multi-agent setup and validation

**Notebook 4 of 5.** Instantiates the Contoso multi-agent system, validates that each
specialist agent can retrieve from its knowledge base, then verifies that the
WorkflowBuilder routing sends queries to the correct specialist.

## Architecture

```
User query
     ↓
 orchestrator           (classifies: HR | MARKETING | PRODUCTS)
     ↓
 ┌───┼──────────┐
 │             │
hr-agent  marketing-agent  products-agent
     ↓             ↓              ↓
 contoso-kb-hr  contoso-kb-marketing  contoso-kb-products
 (answerSynthesis / low effort)
```

## Prerequisites

1. **Run `11-03-knowledge-base-setup.ipynb`** - all KBs and MCP connections must exist.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - run `az login` before executing cells.

## Imports and configuration

In [1]:
import os
import sys
import asyncio
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

# Make local agents module importable
lab_dir = Path.cwd()
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

PROJECT_ENDPOINT = os.environ['CONTOSO_FOUNDRY_PROJECT_ENDPOINT']
SEARCH_ENDPOINT  = os.environ['CONTOSO_SEARCH_ENDPOINT']
CHAT_MODEL       = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')
APIM_CONNECTION  = os.environ['CONTOSO_APIM_CONNECTION']

credential = DefaultAzureCredential()

print(f'Project endpoint : {PROJECT_ENDPOINT}')
print(f'Search endpoint  : {SEARCH_ENDPOINT}')
print(f'Chat model       : {CHAT_MODEL}')
print(f'APIM connection  : {APIM_CONNECTION}')

Project endpoint : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/contoso-project
Search endpoint  : https://contoso-search-n5d3ja.search.windows.net
Chat model       : gpt-4.1-mini
APIM connection  : contoso-apim-connection


## Instantiate agents

Each specialist agent is backed by a `FoundryChatClient` (Foundry Responses API,
inference routed through the project's APIM connection) and an
`AzureAISearchContextProvider` wired to its domain knowledge base.

In [2]:
from agents import (
    create_hr_agent,
    create_marketing_agent,
    create_products_agent,
    create_orchestrator_agent,
    build_contoso_workflow,
)

hr_agent        = create_hr_agent(PROJECT_ENDPOINT, SEARCH_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)
marketing_agent = create_marketing_agent(PROJECT_ENDPOINT, SEARCH_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)
products_agent  = create_products_agent(PROJECT_ENDPOINT, SEARCH_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)
orchestrator    = create_orchestrator_agent(PROJECT_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)

print('Agents instantiated:')
for agent in [orchestrator, hr_agent, marketing_agent, products_agent]:
    print(f'  {agent.name}')

Agents instantiated:
  contoso-orchestrator
  contoso-hr-agent
  contoso-marketing-agent
  contoso-products-agent


---
## Phase 1: Validate specialist agents

Run one validation query directly against each specialist to confirm KB retrieval
is working before testing end-to-end routing.

In [3]:
from display_helpers import show_routing_decision, show_agent_response

SPECIALIST_QUERIES = [
    (hr_agent,        'hr',        'What is the Contoso remote work policy?'),
    (marketing_agent, 'marketing', 'What channels does the Summer Sale 2025 campaign use?'),
    (products_agent,  'products',  'What are the key specs of the ContosoBook Pro?'),
]

for agent, domain, query in SPECIALIST_QUERIES:
    show_routing_decision(query, agent.name)
    response = await agent.run(query)
    show_agent_response(query, response.text, agent.name)
    print()

The Contoso remote work policy allows employees to work from approved locations outside the primary office with manager approval and role eligibility. Employees must have a secure, distraction-free workspace with reliable internet of at least 25 Mbps. Core working hours are from 10:00 AM to 3:00 PM local time for synchronous collaboration. Remote workers must attend all scheduled team meetings via video and keep their calendars updated. Contoso provides necessary equipment like laptops, which employees must care for and return upon separation, and personal use of company equipment is prohibited. Data security requirements include mandatory VPN use. Employees can work remotely up to five days per week or on a hybrid basis, but new employees must be on-site for their first 30 days. Remote work privileges may be revoked if performance, collaboration, or security standards are not met. International remote work requires prior HR and Legal approval due to tax and compliance considerations. This policy is reviewed and updated annually.

The Summer Sale 2025 campaign uses distribution through contoso.com, authorized retail partners, and the Contoso for Business portal. It also employs digital marketing via paid search and shopping ads, LinkedIn, Instagram, YouTube, and display retargeting, along with in-store point-of-sale materials in 320 retail partner locations. This information is from the "Summer Sale Campaign 2025" document.

The key specs of the ContosoBook Pro are:

- Processor: Intel Core Ultra 7 Series 2 with Contoso ContextSense NPU (up to 48 TOPS AI compute)
- Display: 14-inch 2880×1800 OLED, 120Hz refresh rate, 400 nits brightness, 100% DCI-P3 color
- Memory: 16GB or 32GB LPDDR5x
- Storage: 512GB, 1TB, or 2TB NVMe PCIe Gen 4 SSD
- Battery life: Up to 18 hours video playback, 80W Thunderbolt 4 charging
- Ports: 2× Thunderbolt 4, 2× USB-A 3.2 Gen 2, HDMI 2.1, SD card reader, 3.5mm combo audio jack
- Connectivity: Wi-Fi 7, Bluetooth 5.4
- Build: Aircraft-grade aluminum chassis, weighing 1.28 kg
- Keyboard: Full-size island layout, 1.5mm key travel, backlit keys, fingerprint reader in power button, IR camera for Windows Hello
- Dimensions: 311mm × 222mm × 14.8mm
- Operating System: Windows 11 Pro
- Colors: Platinum Silver, Midnight Black
- Warranty: One-year Contoso Premier Support, optional three-year on-site warranty
- Price: Starting at $1,299 USD

SKU: CBP-14-001 [ref_id:0].

---
## Phase 2: Build the workflow

In [4]:
workflow = build_contoso_workflow(
    orchestrator=orchestrator,
    hr_agent=hr_agent,
    marketing_agent=marketing_agent,
    products_agent=products_agent,
)
print(f'Workflow built: {workflow}')

Workflow built: <agent_framework._workflows._workflow.Workflow object at 0x72d6284e5010>


---
## Phase 3: Validate routing

Run three queries through the full workflow and confirm each is routed to the correct
specialist.

In [5]:
from agent_framework import WorkflowAgent

# Wrap the workflow as a WorkflowAgent so it can be invoked like a regular agent
wa = WorkflowAgent(workflow, name='contoso-workflow')

ROUTING_QUERIES = [
    ('HR',        'What are the PTO accrual rates at Contoso for employees with 5 years of tenure?'),
    ('Marketing', 'What is the target ROAS for the Summer Sale 2025 paid channels?'),
    ('Products',  'How long does the ContosoWatch Series 4 battery last with AoD enabled?'),
]

for expected_domain, query in ROUTING_QUERIES:
    response = await wa.run(query)
    print(f'[{expected_domain}] {query}')
    print(f'→ {response.text[:200]}...' if len(response.text) > 200 else f'→ {response.text}')
    print()

[HR] What are the PTO accrual rates at Contoso for employees with 5 years of tenure?
→ At Contoso, employees with 5 years of tenure accrue Paid Time Off (PTO) at a rate of 20 days per year, equivalent to approximately 1.67 days per month. This is specified in the "Paid Time Off Policy" ...

[Marketing] What is the target ROAS for the Summer Sale 2025 paid channels?
→ The target ROAS for the Summer Sale 2025 paid channels is 4.2x.

[Products] How long does the ContosoWatch Series 4 battery last with AoD enabled?
→ The ContosoWatch Series 4 battery lasts 36 hours with the Always on Display (AoD) enabled.



---
## Done

The Contoso multi-agent system is validated:

| Agent | Knowledge Base | Status |
|-------|---------------|--------|
| `contoso-hr-agent` | `contoso-kb-hr` | ✓ Validated |
| `contoso-marketing-agent` | `contoso-kb-marketing` | ✓ Validated |
| `contoso-products-agent` | `contoso-kb-products` | ✓ Validated |
| `contoso-orchestrator` | - | ✓ Routing verified |

**Next step:** run `11-05-multi-agent-queries.ipynb` for the full routing demo with
`WorkflowBuilder` visualisations and citation display.